In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/telco_churn.csv")
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

numeric_features = X.select_dtypes(
    include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Treino:", X_train.shape, "| Teste:", X_test.shape)

Treino: (5634, 19) | Teste: (1409, 19)


C:\Users\denar\AppData\Local\Temp\ipykernel_49168\1777747673.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, roc_auc_score, classification_report

# Igual ao baseline: numéricas normalizadas, categóricas com One-Hot
preprocessor_mlp = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

mlp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_mlp),
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(50, 25),  # duas camadas ocultas: 50 e 25 neurônios
        activation='relu',             # função de ativação padrão, eficiente e comum
        max_iter=500,                  # número máximo de iterações de treino
        early_stopping=True,           # para automaticamente se não houver melhora
        random_state=42
    ))
])

mlp_pipeline.fit(X_train, y_train)

y_pred_mlp = mlp_pipeline.predict(X_test)
y_proba_mlp = mlp_pipeline.predict_proba(X_test)[:, 1]

f1_mlp = f1_score(y_test, y_pred_mlp)
auc_mlp = roc_auc_score(y_test, y_proba_mlp)

print(f"F1-score: {f1_mlp:.4f}")
print(f"AUC-ROC: {auc_mlp:.4f}")
print("\nRelatório completo:")
print(classification_report(y_test, y_pred_mlp,
      target_names=['No Churn', 'Churn']))

F1-score: 0.5701
AUC-ROC: 0.8430

Relatório completo:
              precision    recall  f1-score   support

    No Churn       0.83      0.90      0.87      1035
       Churn       0.65      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.79      0.80      0.79      1409

